**Part 2 · LangChain** · step 4 of 23  
[← 03 Python prerequisites](../01_AI_Fundamentals/03_python_prerequisites.ipynb) · [🏠 Roadmap](../README.md) · [05 Messages →](05_messages.ipynb)

# 04 · Chat Models & Providers — OpenAI, Google Gemini, Groq

## Why LangChain?
In notebooks 01–02 you called OpenAI with its own SDK. That works, but:
- every provider (OpenAI, Google, Groq, Anthropic…) has a **different SDK and response format**;
- tool calling, structured output, streaming and retries all need **boilerplate** you rewrite each time.

**LangChain** gives one standard interface over all providers plus ready-made building blocks:
models → messages → prompts → tools → structured output → agents → middleware.
Switching from OpenAI to Groq becomes a one-string change.

**What you'll learn**
- Two ways to create a chat model:
  1. `init_chat_model("provider:model")` — one generic factory for every provider.
  2. Provider classes — `ChatOpenAI`, `ChatGoogleGenerativeAI`, `ChatGroq` — for provider-specific options.
- Three ways to call a model: **`invoke`** (one answer), **`stream`** (token by token), **`batch`** (many inputs in parallel).

Every chat model shares the same interface, so you can swap providers without changing the rest of your code.

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

## 1. `init_chat_model` — the provider-agnostic factory

In [ ]:
from langchain.chat_models import init_chat_model

# Format: "<provider>:<model-name>"
model = init_chat_model("openai:gpt-4.1")
resp = model.invoke("Hello how are you")
resp.content  # invoke() returns an AIMessage; .content is the text

In [ ]:
# Google Gemini (needs GOOGLE_API_KEY)
gemini = init_chat_model("google_genai:gemini-3.5-flash")
resp = gemini.invoke("what is 2+2")
resp.content

In [ ]:
# Groq (needs GROQ_API_KEY) — very fast inference for open-source models
groq = init_chat_model("groq:qwen/qwen3-32b")
resp = groq.invoke("why do parrots talk?")
resp.content

## 2. Provider-specific classes
Same result as above, but you import the integration package directly.
Use these when you need provider-only parameters.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

openai_model = ChatOpenAI(model="gpt-4.1")
gemini_model = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
groq_model = ChatGroq(model="qwen/qwen3-32b")

openai_model.invoke("Hello how are you")

> **Note:** a `503 UNAVAILABLE` from Gemini means the model is overloaded on Google's side — just retry later.

## 3. Invoke vs Stream vs Batch

| Method | Returns | Use when |
|---|---|---|
| `invoke(input)` | one `AIMessage` | you just need the answer |
| `stream(input)` | iterator of `AIMessageChunk` | show text as it is generated (chat UIs) |
| `batch([inputs], config={"max_concurrency": n})` | list of `AIMessage` | many **independent** requests — runs them in parallel |

In [ ]:
model = groq_model
model.invoke("write me a 200 words paragraph on AI")

### Streaming — print each chunk as it arrives

In [ ]:
for chunk in model.stream("write me a 200 words paragraph on AI"):
    print(chunk.text, end=" ", flush=True)

 <think> 
 Okay ,  I  need  to  write  a   2 0 0 -word  paragraph  about  AI .  Let  me  start  by  thinking  about  the  key  points .  AI  is  a  broad  topic ,  so  maybe  focus  on  its  applications ,  benefits ,  and  challenges .  First ,  I  should  define  what  AI  is — maybe  mention  machine  learning  and  neural  networks .  Then  talk  about  different  fields  where  AI  is  used ,  like  healthcare ,  finance ,  and  transportation .  Examples  could  include  diagnostics ,  fraud  detection ,  autonomous  vehicles .  Also ,  mention  natural  language  processing  with  chat bots  and  virtual  assistants .  Then  touch  on  the  benefits :  efficiency ,  accuracy ,  solving  complex  problems .  But  also  address  the  challenges :  ethical  concerns ,  job  displacement ,  data  privacy .  Maybe  mention  the  importance  of  regulations  and  responsible  development .  Need  to  ensure  the  paragraph  flows  well ,  connecting  each  idea .  Avoid  technical  j 

In [ ]:
# Using "|" as the separator makes the individual chunks (tokens) visible
for chunk in model.stream("why do parrots have colourful feathers?"):
    print(chunk.text, end="|", flush=True)

|<think>|
|Okay|,| so| I| need| to| figure| out| why| par|rots| have| colorful| feathers|.| Let| me| start| by| recalling| what| I| know| about| par|rots|.| They|'re| known| for| their| vibrant| colors|—|greens|,| blues|,| red|s|,| yell|ows|,| etc|.| I| remember| reading| that| color|ation| in| animals| can| serve| various| purposes| like| camouflage|,| communication|,| or| attracting| mates|.| But| par|rots| are| usually| very| visible|,| so| maybe| their| colors| aren|'t| for| hiding|.| 

|First|,| I| should| think| about| sexual| selection|.| In| many| bird| species|,| males| have| brighter| colors| to| attract| females|.| Maybe| female| par|rots| prefer| males| with| more| vibrant| feathers|,| so| males| have| evolved| to| have| these| colors| to| increase| their| chances| of| mating|.| That| makes| sense|.| But| I| should| check| if| this| applies| to| par|rots| specifically|.| I| think| some| par|rot| species| are| mon|og|amous|,| but| does| that| affect| their| color|ation|?| Ma

### Batch — many independent requests in parallel
Batching can significantly improve throughput and reduce cost because the requests are processed concurrently.

In [ ]:
responses = model.batch(
    [
        "why do parrots have colorful feathers?",
        "How do airplanes fly?",
        "what is quantum computing?",
    ],
    config={"max_concurrency": 5},  # at most 5 calls in flight at once
)

for res in responses:
    print(res)

content="<think>\nOkay, so the user is asking why parrots have colorful feathers. Let me start by recalling what I know about parrots and their feathers. I remember reading that many birds have bright plumage for specific reasons, like attracting mates or communication. Parrots are known for their vibrant colors, so maybe similar reasons apply here.\n\nFirst, I should consider sexual selection. In many bird species, males have brighter colors to attract females. Parrots are mostly monogamous, so maybe the females choose mates based on the brightness or specific colors of their feathers. That could be a key point.\n\nThen there's the aspect of camouflage. Wait, but parrots are usually in tropical environments with lots of green foliage. If they had colors that blended in, maybe they'd be safer from predators. But their feathers are so bright. Maybe they don't rely on camouflage. Instead, their colors might serve another purpose. Maybe the colors are a sign of health or genetic fitness. 

## Revision notes
- `init_chat_model("openai:gpt-4.1")`, `"google_genai:..."`, `"groq:..."` — one factory, many providers.
- All chat models return **`AIMessage`** from `invoke`.
- `stream` → chunks; `batch` → list; `max_concurrency` limits parallelism.
- Reasoning models like Qwen3 print a `<think>` block before the answer — that's the model's reasoning, not a bug.

---
✅ **Done with 04 Chat models & providers.** Next: [05 Messages →](05_messages.ipynb)  
[🏠 Roadmap](../README.md)